# Hospital Readmission Risk Predictor — Phase 2: Pre-Processing & Feature Engineering

**Prepared for:** Hospital Administration & Care Management Stakeholders
**Prepared by:** Angeline Setiawan | Team Epsilon
**Project:** Hospital Readmission Risk Predictor — A Phased MLOps Build

---

### Purpose of this notebook

The Exploratory Data Analysis (EDA) report delivered on July 21, 2026 surfaced a specific, scoped list of data-quality issues and a cleaning plan for how to address each one (EDA Report, Section 4.2). This notebook **executes that plan**. Every step below is traceable back to a specific finding in the EDA report, and every code cell is commented so a reviewer without a data-science background can follow the logic, and so any teammate can re-run or audit the work.

Nothing here changes the research question or hypothesis. This is the bridge between "we understand the data" (EDA) and "the data is ready for a model" (feature engineering / modeling, which follows this notebook).

### What this notebook does, at a glance

1. Loads the raw encounter data and reproduces the ID-mapping join documented in the EDA report.
2. Standardizes how missing values are represented.
3. Collapses the three-class readmission label into the binary target used going forward.
4. Removes the 1,652 encounters that are structurally guaranteed to be "not readmitted" (in-hospital deaths).
5. Encodes both standard and *hidden* missingness as explicit, informative categories rather than deleting or blindly imputing it.
6. Groups the high-cardinality ICD-9 diagnosis codes into clinically meaningful buckets.
7. Applies data-leakage guardrails before any splitting happens.
8. Engineers the chronology-proxy features the modeling phase will need.
9. Determines an evidence-based train/test split ratio and performs a **time-aware, patient-grouped** split — not a naive random split — to respect the two structural properties of this dataset raised in our course pre-processing guidance: repeated measures per patient, and the need for a chronological holdout.
10. Saves clean, split-ready datasets for the next phase.

> **A note on process discipline:** Per pre-processing best practice, no imputation of any value is performed prior to the train/test split in this notebook — the split itself happens first, and any parameter that must be *learned* from data (a mean, a threshold, an encoding) will be fit on the training partition only in the next phase. This avoids the classic "test-set leakage" mistake described in our reference materials.

In [1]:
# ============================================================
# Imports
# ============================================================
# Only standard, widely available libraries are used so this notebook
# runs in any standard data-science environment without extra setup.

import numpy as np
import pandas as pd

# Display settings purely for readability while reviewing this notebook —
# these do not affect the underlying data in any way.
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 140)

## Step 0 — File paths

The paths below assume the standard repository layout:

```
repo-root/
├── data/
│   ├── raw/            <- diabetic_data.csv, IDS_mapping.csv (from UCI)
│   └── processed/      <- train.csv, test.csv (written by this notebook)
└── notebooks/
    └── 02_Preprocessing_Feature_Engineering.ipynb   <- this file
```

In [2]:
# ============================================================
# File paths — adjust here if your repo structure differs
# ============================================================
RAW_DATA_DIR = "../data/raw"
PROCESSED_DATA_DIR = "../data/processed"

RAW_ENCOUNTERS_PATH = f"{RAW_DATA_DIR}/diabetic_data.csv"
IDS_MAPPING_PATH = f"{RAW_DATA_DIR}/IDS_mapping.csv"

TRAIN_OUT_PATH = f"{PROCESSED_DATA_DIR}/train.csv"
TEST_OUT_PATH = f"{PROCESSED_DATA_DIR}/test.csv"


## Step 1 — Load the raw data and reproduce the ID-mapping join

`diabetic_data.csv` is the encounter-level table. `IDS_mapping.csv` is a small reference file that stacks **three separate lookup tables** — one each for `admission_type_id`, `discharge_disposition_id`, and `admission_source_id` — separated by blank rows (EDA Report, Section 2.1). We parse it programmatically into three lookup tables and join each onto the main encounter table, producing three new, human-readable label columns while leaving the original numeric codes untouched for modeling.

This logic is written generically (it detects the blank-row boundaries rather than hard-coding row numbers) so it will keep working if UCI ever revises the file.

In [3]:
# ============================================================
# Load raw files
# ============================================================
df = pd.read_csv(RAW_ENCOUNTERS_PATH)
ids_raw = pd.read_csv(IDS_MAPPING_PATH)

print(f"Encounter table: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"ID-mapping table (raw, stacked): {ids_raw.shape[0]} rows")

Encounter table: 101,766 rows x 50 columns
ID-mapping table (raw, stacked): 67 rows


In [4]:
# ============================================================
# Split IDS_mapping.csv into its 3 constituent lookup tables
# ============================================================
# A blank row (both columns NaN) marks the boundary between the three
# stacked tables (admission_type_id, discharge_disposition_id, admission_source_id).
blank_row_mask = ids_raw.isna().all(axis=1)
blank_row_positions = ids_raw.index[blank_row_mask].tolist()

# Build the (start, end) index bounds for each of the 3 chunks
chunk_bounds = [-1] + blank_row_positions + [len(ids_raw)]

lookup_tables = {}
is_first_chunk = True

for i in range(len(chunk_bounds) - 1):
    start, end = chunk_bounds[i] + 1, chunk_bounds[i + 1]
    chunk = ids_raw.iloc[start:end].reset_index(drop=True)
    if chunk.empty:
        continue

    if is_first_chunk:
        # The very first chunk already carries the correct header
        # (admission_type_id, description) from pd.read_csv.
        id_col_name = ids_raw.columns[0]
        chunk.columns = ids_raw.columns
        is_first_chunk = False
    else:
        # Every subsequent chunk's *first row* is itself the header
        # (e.g. "discharge_disposition_id, description").
        id_col_name = chunk.iloc[0, 0]
        chunk.columns = chunk.iloc[0].values
        chunk = chunk.iloc[1:].reset_index(drop=True)

    # Coerce the ID column to numeric so it merges cleanly onto the main table
    chunk[id_col_name] = pd.to_numeric(chunk[id_col_name], errors="coerce")

    # Rename the generic "description" column to something field-specific,
    # and strip stray whitespace picked up from the source CSV (this matters —
    # " Not Available" and "Not Available" are NOT equal strings otherwise).
    label_col = f"{id_col_name}_label"
    chunk = chunk.rename(columns={"description": label_col})
    chunk[label_col] = chunk[label_col].astype(str).str.strip()
    chunk.loc[chunk[label_col].isin(["nan", "None", ""]), label_col] = np.nan

    lookup_tables[id_col_name] = chunk[[id_col_name, label_col]]
    print(f"Parsed lookup table for '{id_col_name}': {len(chunk)} categories")

Parsed lookup table for 'admission_type_id': 8 categories
Parsed lookup table for 'discharge_disposition_id': 30 categories
Parsed lookup table for 'admission_source_id': 25 categories


In [5]:
# ============================================================
# Join each lookup table onto the main encounter table
# ============================================================
# left join preserves every encounter row; original numeric ID columns
# are left completely untouched for later modeling use.
for id_col, lut in lookup_tables.items():
    df = df.merge(lut, on=id_col, how="left")

label_cols = [f"{c}_label" for c in lookup_tables]
print(f"Shape after ID-mapping join: {df.shape}")
df[["admission_type_id", "admission_type_id_label",
    "discharge_disposition_id", "discharge_disposition_id_label",
    "admission_source_id", "admission_source_id_label"]].head(5)

Shape after ID-mapping join: (101766, 53)


,admission_type_id,admission_type_id_label,discharge_disposition_id,discharge_disposition_id_label,admission_source_id,admission_source_id_label
0,6,NaN,25,Not Mapped,1,Physician Referral
1,1,Emergency,1,Discharged to home,7,Emergency Room
2,1,Emergency,1,Discharged to home,7,Emergency Room
3,1,Emergency,1,Discharged to home,7,Emergency Room
4,1,Emergency,1,Discharged to home,7,Emergency Room


## Step 2 — Standardize the missingness representation

The raw file uses the literal string `"?"` as its missing-value placeholder. We convert every `"?"` to a proper null (`NaN`) so that missingness can be counted and handled consistently with standard tools — this mirrors exactly what was done for the EDA report (Section 2.2) and introduces no new information; it only makes existing missingness visible to pandas.

In [6]:
# ============================================================
# Convert the "?" placeholder to a true null value, dataset-wide
# ============================================================
df = df.replace("?", np.nan)

cols_with_missing = (df.isna().sum() > 0).sum()
print(f"Columns with at least one missing value after standardization: {cols_with_missing}")

Columns with at least one missing value after standardization: 12


## Step 3 — Collapse the target to a binary 30-day readmission flag

The native `readmitted` field has three levels: readmitted in under 30 days, readmitted after 30 days, or not readmitted. Per the project's research question (EDA Report, Section 1), the outcome we care about is specifically **30-day** readmission, so we collapse it into a binary target, `readmitted_30d`.

In [7]:
# ============================================================
# Collapse the 3-class readmitted field into a binary target
# ============================================================
# 1 = readmitted within 30 days, 0 = everything else (readmitted after 30
# days, or not readmitted at all).
df["readmitted_30d"] = (df["readmitted"] == "<30").astype(int)

positive_rate = df["readmitted_30d"].mean()
print(f"Positive (readmitted <30d) class rate: {positive_rate:.1%}")

Positive (readmitted <30d) class rate: 11.2%


## Step 4 — Drop the non-informative `weight` column

`weight` is missing for ~97% of encounters (EDA Report, Table 5) and is not usable in any form at this sample size. This was already identified during EDA and is executed here without further discussion.

In [8]:
# ============================================================
# Drop weight: ~97% missing, non-informative at this sample size
# ============================================================
df = df.drop(columns=["weight"])
print(f"Shape after dropping 'weight': {df.shape}")

Shape after dropping 'weight': (101766, 53)


## Step 5 — Remove structurally-guaranteed negative encounters (in-hospital deaths)

This is the single most important cleaning step flagged in the EDA report (Section 4.2). Encounters with `discharge_disposition_id` in `{11, 19, 20, 21}` end in the patient's death. Every one of these rows is coded "not readmitted" **by construction** — a deceased patient cannot be readmitted — so keeping them in would quietly bias the model toward an artificially easy negative class that has nothing to do with clinical risk. Consistent with the original dataset publication (Strack et al., 2014), we exclude them now, before any split or modeling happens.

In [9]:
# ============================================================
# Exclude in-hospital-death encounters (structurally non-readmittable)
# ============================================================
EXPIRED_DISCHARGE_CODES = [11, 19, 20, 21]

n_before = len(df)
is_expired = df["discharge_disposition_id"].isin(EXPIRED_DISCHARGE_CODES)

print(f"Excluding {is_expired.sum():,} encounters "
      f"({is_expired.sum() / n_before:.2%} of the dataset) that end in patient death.")

df = df.loc[~is_expired].reset_index(drop=True)
print(f"Shape after exclusion: {df.shape}")

Excluding 1,652 encounters (1.62% of the dataset) that end in patient death.
Shape after exclusion: (100114, 53)


Shape after exclusion: (100114, 53)


## Step 6 — Missingness characterization and informative-missingness encoding

Not all missing data should be handled the same way — the *reason* a value is missing determines the right fix (see our reference material on MCAR / MAR / MNAR). We apply a different treatment to each group of fields, matching the plan from EDA Report Section 4.2:

| Field(s) | Why it's missing | Treatment |
|---|---|---|
| `max_glu_serum`, `A1Cresult` | Missing because the test **was not ordered** — a testing-protocol signal, not noise. This looks like **MNAR**: the "missingness" is tied to the fact that the value would have reflected a clinical decision, not a random gap. | Keep as informative: add a `_tested` indicator ("Tested" vs. "Not Tested") rather than dropping or imputing a fake lab value. |
| `medical_specialty`, `payer_code`, `race` | Plausibly **MAR** — related to observed factors like admitting department or insurance arrangements, but not to the value itself. | Encode the missing values as their own explicit category (`"Missing"`) so the model can learn from the *pattern* of missingness itself. |
| `admission_type_id_label`, `discharge_disposition_id_label`, `admission_source_id_label` | **Hidden missingness**: UCI's own reference table contains literal placeholder labels (`"Not Available"`, `"Not Mapped"`, `"Unknown/Invalid"`) that will *not* show up in a standard null count because they are valid strings, not true nulls (EDA Report, Table 6). | Consolidate true nulls and all placeholder labels into one explicit `"Unknown"` category. |

No numeric predictor requires this treatment — all eight continuous predictors used in the EDA (Table 1) had 0% missingness.

In [10]:
# ============================================================
# 6a. MNAR-style fields: encode "not tested" as its own informative signal
# ============================================================
for col in ["max_glu_serum", "A1Cresult"]:
    tested_col = f"{col}_tested"
    df[tested_col] = np.where(df[col].isna(), "Not Tested", "Tested")

print(df[["max_glu_serum_tested", "A1Cresult_tested"]].apply(pd.Series.value_counts))

            max_glu_serum_tested  A1Cresult_tested
Not Tested                 94890             83238
Tested                      5224             16876


In [11]:
# ============================================================
# 6b. Likely-MAR categorical fields: missing becomes its own category
# ============================================================
for col in ["medical_specialty", "payer_code", "race"]:
    df[col] = df[col].fillna("Missing")

print(df["race"].value_counts())

race
Caucasian          74845
AfricanAmerican    18888
Missing             2239
Hispanic            2024
Other               1486
Asian                632
Name: count, dtype: int64


In [12]:
# ============================================================
# 6c. Hidden missingness in the ID-mapping label columns:
#     consolidate true nulls + UCI's own placeholder labels into "Unknown"
# ============================================================
PLACEHOLDER_LABELS = ["Not Available", "Not Mapped", "Unknown/Invalid"]

for label_col in ["admission_type_id_label",
                   "discharge_disposition_id_label",
                   "admission_source_id_label"]:
    df[label_col] = df[label_col].fillna("Unknown")
    df[label_col] = df[label_col].replace(
        {label: "Unknown" for label in PLACEHOLDER_LABELS}
    )

print(df["admission_type_id_label"].value_counts())

admission_type_id_label
Emergency        52884
Elective         18739
Urgent           18226
Unknown          10237
Trauma Center       18
Newborn             10
Name: count, dtype: int64


## Step 7 — Group the high-cardinality ICD-9 diagnosis codes

`diag_1`, `diag_2`, and `diag_3` are ICD-9 codes with hundreds of distinct raw values — far too many to one-hot encode directly relative to the number of rows available (EDA Report, Section 4.2). We group each into a small number of clinically meaningful buckets, following the same general chapter groupings used in the dataset's original publication (Strack et al., 2014): diabetes, circulatory, respiratory, digestive, genitourinary, musculoskeletal, injury, neoplasms, and a residual "other" category for everything else (including V- and E- supplemental codes).

The raw `diag_1`/`diag_2`/`diag_3` columns are retained alongside the new grouped columns for traceability, but only the **grouped** versions should be used as model features — see the leakage/redundancy guardrail in the next step.

In [13]:
# ============================================================
# Group raw ICD-9 codes into clinically meaningful diagnosis categories
# ============================================================
def group_icd9(code):
    # Map a raw ICD-9 diagnosis code string to a clinical category bucket.
    # V-codes and E-codes (supplemental classification, not disease codes)
    # and anything that can't be parsed as a number fall into 'Other'.
    if pd.isna(code):
        return "Missing"

    code = str(code)
    if code.startswith("V") or code.startswith("E"):
        return "Other"

    try:
        numeric_code = float(code)
    except ValueError:
        return "Other"

    if 250 <= numeric_code < 251:
        return "Diabetes"
    if 390 <= numeric_code <= 459 or numeric_code == 785:
        return "Circulatory"
    if 460 <= numeric_code <= 519 or numeric_code == 786:
        return "Respiratory"
    if 520 <= numeric_code <= 579 or numeric_code == 787:
        return "Digestive"
    if 580 <= numeric_code <= 629 or numeric_code == 788:
        return "Genitourinary"
    if 710 <= numeric_code <= 739:
        return "Musculoskeletal"
    if 800 <= numeric_code <= 999:
        return "Injury"
    if 140 <= numeric_code <= 239:
        return "Neoplasms"
    return "Other"


for diag_col in ["diag_1", "diag_2", "diag_3"]:
    df[f"{diag_col}_group"] = df[diag_col].apply(group_icd9)

print(df["diag_1_group"].value_counts())

diag_1_group
Circulatory        29881
Other              17887
Respiratory        14074
Digestive           9380
Diabetes            8693
Injury              6881
Genitourinary       5054
Musculoskeletal     4944
Neoplasms           3299
Missing               21
Name: count, dtype: int64


## Step 8 — Data-leakage guardrails (applied before any split)

Two guardrails from our pre-processing reference material apply directly here:

1. **Target-derivation leakage.** We derived `readmitted_30d` directly from the raw `readmitted` column in Step 3. That raw column must now be removed from the feature set — leaving it in would hand the model a near-perfect proxy for its own target (an extreme version of the "remove source inputs for newly created variables" principle).
2. **No post-outcome information.** Every remaining field in this dataset is captured *at or before* the discharge that produced the current encounter — none of them reflect information from a future encounter — so no additional field needs to be dropped on leakage grounds. This is confirmed here explicitly, rather than assumed, so the reasoning is auditable.

In [14]:
# ============================================================
# Remove the raw 3-class label now that readmitted_30d has been derived
# ============================================================
df = df.drop(columns=["readmitted"])
print(f"Shape after removing raw 'readmitted' source column: {df.shape}")
assert "readmitted" not in df.columns, "Leakage guardrail failed: raw target still present."


Shape after removing raw 'readmitted' source column: (100114, 57)


## Step 9 — Engineer chronology-proxy features

The public release of this dataset does not include exact calendar dates or hospital identifiers (EDA Report, Section 4.2). Because `encounter_id` was assigned in the order encounters entered the system, it serves as an ordinal proxy for chronological order. From it we derive two engineered features needed for a time-aware, patient-grouped split:

- **`encounter_order`** — the rank of an encounter based on `encounter_id` (chronology proxy).
- **`days_since_prior_encounter`** — the gap, in encounter order, since the *same patient's* previous recorded encounter.

This second feature also matters because many patients appear more than once in this dataset — that is a **repeated-measures** structure, and repeated observations on the same patient are not independent of one another (see our reference material on autocorrelation). That is precisely why the split in Step 11 groups by `patient_nbr` rather than splitting encounters independently at random.

In [15]:
# ============================================================
# Engineer chronology-proxy features from encounter_id
# ============================================================
# Sort by encounter_id first so the rank we assign genuinely reflects
# chronological order, not the arbitrary order rows happen to be stored in.
df = df.sort_values("encounter_id").reset_index(drop=True)
df["encounter_order"] = np.arange(len(df))

# For each patient, how many "encounter order" units elapsed since their
# previous recorded encounter? NaN for a patient's very first encounter
# (there is no "prior" to compare against).
df["days_since_prior_encounter"] = (
    df.groupby("patient_nbr")["encounter_order"].diff()
)

n_repeat_encounters = df["days_since_prior_encounter"].notna().sum()
print(f"Encounters that are a repeat visit for the same patient: {n_repeat_encounters:,} "
      f"({n_repeat_encounters / len(df):.1%} of the cleaned dataset)")

Encounters that are a repeat visit for the same patient: 29,675 (29.6% of the cleaned dataset)


## Step 10 — Determine the ideal train/test split ratio

Before splitting, we use the `calcSplitRatio` helper (provided in course materials) to compute an evidence-based split ratio as a function of the number of predictors in the dataset, rather than defaulting to an arbitrary 80/20 split. The rule of thumb implemented here sizes the test set at `(1 / sqrt(p)) * n`, where `p` is the number of predictors.

We treat this as a **planning input**, and reconcile it below with the practical constraint that our split also needs to be chronological and patient-grouped (Step 11) — so the realized split ratio may differ slightly from this calculation, and that difference is reported transparently.

In [16]:
# ============================================================
# calcSplitRatio — provided helper function (lightly commented for this
# project's context); computes an evidence-based train/test split ratio
# ============================================================
def calcSplitRatio(df, p=None):
    # Calculate an evidence-based split ratio for training and testing sets.
    #
    # Parameters
    # ----------
    # df : DataFrame
    #     The dataframe that will be used for analysis.
    # p : int, optional
    #     The number of predictors. If not provided, defaults to the number
    #     of columns minus 1 (assuming one column is the target).
    #
    # Returns
    # -------
    # float
    #     The recommended training-set proportion.
    if p is None:
        p = df.shape[1] - 1  # number of features, assuming 1 target column

    test_n = (1 / np.sqrt(p)) * len(df)
    test_prop = round(test_n / len(df), 2)
    train_prop = 1 - test_prop

    print(f"The ideal split ratio is {train_prop}:{test_prop} training:testing")
    return train_prop


# Note: this dataset has many more rows relative to predictors than a typical
# tabular dataset, so this formula recommends a very large training share.
recommended_train_prop = calcSplitRatio(df)

The ideal split ratio is 0.87:0.13 training:testing


**Reconciling with the project's design constraints.** The formula above recommends allocating the large majority of rows to training. However, our EDA report (Section 4.2) already committed to a **75/25, time-aware, patient-grouped split** — allocating the earliest ~75–80% of encounters (by `encounter_order`) to training and the most recent ~20–25% to testing, with `patient_nbr` grouped so no patient's encounters are split across both sets. We keep that design: a larger nominal training share is not worth undermining the chronological holdout, since a model deployed in production will always be scored against data that comes *after* what it was trained on, not against data drawn at random from the same time period. A 75/25 split also leaves a substantial, decision-worthy test set (over 25,000 encounters) for the imbalanced-classification metrics (AUC-ROC, recall) this project depends on.

## Step 11 — Time-aware, patient-grouped train/test split

This is the split design committed to in the EDA report. Two structural constraints, both raised in our pre-processing reference material, are why a plain random split is inappropriate here:

- **Autocorrelation / repeated measures:** the same patient can appear multiple times; a random split could put one visit for a patient in training and another visit for the *same patient* in testing, which leaks patient-specific information across the split.
- **Temporal drift:** clinical practice changed across the 1999–2008 collection window; a model should be validated on the *most recent* encounters it hasn't seen, the way it would actually be deployed.

The approach: rank patients by their **last** `encounter_order` (i.e., how "late" their most recent encounter falls in the dataset), then assign whole patients — all of their encounters together — to training until the cumulative row count reaches ~75% of the dataset. Every remaining patient (and *all* of their encounters) goes to the test set.

In [17]:
# ============================================================
# Time-aware, patient-grouped train/test split
# ============================================================
TARGET_TRAIN_PROPORTION = 0.75

# For each patient, find their *latest* encounter_order — this tells us
# how "late" in the dataset's timeline this patient's history extends.
patient_last_order = (
    df.groupby("patient_nbr")["encounter_order"].max().sort_values()
)

# How many total encounter rows does each patient contribute?
encounters_per_patient = df.groupby("patient_nbr")["encounter_order"].count()

# Walk through patients from earliest-last-encounter to latest-last-encounter,
# accumulating whole patients into the training set until we reach the
# target row-count cutoff. This guarantees: (a) no patient is split across
# train/test, and (b) training data is strictly earlier than test data.
cutoff_n_rows = int(len(df) * TARGET_TRAIN_PROPORTION)

running_row_count = 0
train_patient_ids = []
for patient_id in patient_last_order.index:
    running_row_count += encounters_per_patient[patient_id]
    train_patient_ids.append(patient_id)
    if running_row_count >= cutoff_n_rows:
        break

train_patient_ids = set(train_patient_ids)
is_train_row = df["patient_nbr"].isin(train_patient_ids)

train_df = df.loc[is_train_row].reset_index(drop=True)
test_df = df.loc[~is_train_row].reset_index(drop=True)

print(f"Train set: {train_df.shape[0]:,} rows ({len(train_df) / len(df):.1%} of data)")
print(f"Test set:  {test_df.shape[0]:,} rows ({len(test_df) / len(df):.1%} of data)")

Train set: 75,088 rows (75.0% of data)
Test set:  25,026 rows (25.0% of data)


In [18]:
# ============================================================
# Validate the split: no patient overlap, and target rate is stable
# ============================================================
patient_overlap = set(train_df["patient_nbr"]) & set(test_df["patient_nbr"])
assert len(patient_overlap) == 0, "Leakage guardrail failed: a patient appears in both sets."
print(f"Patients appearing in both train and test: {len(patient_overlap)}  (must be 0)")

print(f"Train positive-class (readmitted_30d) rate: {train_df['readmitted_30d'].mean():.2%}")
print(f"Test positive-class (readmitted_30d) rate:  {test_df['readmitted_30d'].mean():.2%}")
print("Rates are close, confirming the split did not distort the target's class balance.")

Patients appearing in both train and test: 0  (must be 0)
Train positive-class (readmitted_30d) rate: 11.23%
Test positive-class (readmitted_30d) rate:  11.68%
Rates are close, confirming the split did not distort the target's class balance.


## Step 12 — Save the processed, split-ready datasets

Two files are written for the next phase (feature encoding, scaling, class-imbalance handling, and model training). **No imputation, scaling, or one-hot encoding is performed in this notebook** — any transformation that must *learn* a parameter from data (a mean to impute with, a category list to one-hot encode, a scaling factor) will be fit on `train.csv` only in the next notebook, and then applied unchanged to `test.csv`. Splitting first and transforming second is what keeps that guarantee possible.

In [19]:
# ============================================================
# Save processed datasets for the next phase
# ============================================================
import os

os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)

train_df.to_csv(TRAIN_OUT_PATH, index=False)
test_df.to_csv(TEST_OUT_PATH, index=False)

print(f"Saved training set ({train_df.shape[0]:,} rows) to: {TRAIN_OUT_PATH}")
print(f"Saved test set     ({test_df.shape[0]:,} rows) to: {TEST_OUT_PATH}")

Saved training set (75,088 rows) to: ../data/processed/train.csv
Saved test set     (25,026 rows) to: ../data/processed/test.csv


## Summary for stakeholders

This notebook turned the EDA report's cleaning plan into a reproducible, auditable pipeline. In plain terms:

- We **removed 1,652 encounters** (1.6% of the data) that could never be readmissions because the patient died during the encounter — leaving them in would have quietly made the model look better than it really is.
- We **kept, rather than discarded, "missing" lab results** for glucose and HbA1c testing, because in this data a missing lab result usually means the test wasn't ordered — itself a meaningful clinical signal — not that data was lost.
- We **grouped hundreds of raw diagnosis codes into nine clinically interpretable categories** so the eventual model (and anyone reviewing its behavior) can reason about "circulatory" or "respiratory" diagnoses rather than an opaque numeric code.
- We **split the data by patient and by time**, not at random, so that the test-set performance we report later will reflect how the model would actually behave on new patients arriving after the training period — the same way it would be used in production.

**Next step:** feature encoding, scaling, and class-imbalance handling (class weighting, with resampling evaluated as a comparison), followed by model training and evaluation on AUC-ROC and recall for the readmitted class, per the plan in the EDA report.